# 거래 분류 에이전트 실거래 테스트 결과

가맹점 이름 목록을 **실제 분류 에이전트**(1~6단계, 실제 SVM · 네이버/카카오 검색 · Jev)로 분류하고 결과를 표로 본다.
내부적으로 `python manage.py classify_names`를 실행한다. 임시 거래로 돌린 뒤 DB는 되돌린다.

- 실거래 이름·결과는 개인 데이터라 `local_workspace/data/classifier/`(Git 제외)에 둔다. **노트북 출력은 지우고 커밋**
- 노트북은 `scripts/` 폴더에서 열고 커널은 저장소의 `.venv`. Docker DB가 켜져 있어야 한다
- 5단계까지 가는 이름은 네이버·Jev를 실제로 호출한다 (상호만 전송). `.env` 키가 없으면 4단계에서 끝난다

In [ ]:
import os
import sys
from io import StringIO
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

ROOT = Path(find_dotenv(usecwd=True)).parent              # 저장소 루트 (.env 가 있는 곳)
sys.path.insert(0, str(ROOT / "backend"))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"          # 주피터는 이벤트 루프 안에서 돌아서 Django DB 호출에 필요
import django
django.setup()
from django.core.management import call_command
from scripts.spending_classifier.common import CATEGORIES

DATA = ROOT / "local_workspace/data/classifier"
NAMES = DATA / "real_merchants.txt"                       # 한 줄에 가맹점 이름 하나
RESULT = DATA / "real_merchants_result.csv"               # 분류 결과 (정답 칸을 채우면 5번에서 정확도 계산)
NAME_OF = {**CATEGORIES, "ETC": "기타", "UNCLASSIFIED": "확인 필요"}
print(NAMES.exists(), RESULT.exists())

## 1. 실제 분류 실행
약 30초~1분. 이미 만든 결과를 보기만 하려면 건너뛰고 2번부터. **다시 실행하면 결과 CSV의 정답 칸도 새로 만들어진다** (정답을 채웠다면 먼저 다른 이름으로 복사)

In [ ]:
out = StringIO()
call_command("classify_names", str(NAMES), "--csv", str(RESULT), stdout=out)
print(out.getvalue().splitlines()[-2])  # 총 N건 요약

## 2. 결과 표
단계: `1 다품목` · `1 페이 충전` · `3 규칙` · `3 이름 모름` · `4 SVM` · `5 검색+Jev`. 결과가 `확인 필요`면 사용자에게 묻는 거래

In [ ]:
df = pd.read_csv(RESULT, encoding="utf-8-sig", dtype=str).fillna("")
df["결과"] = df["결과"].map(lambda c: NAME_OF.get(c, c))
df["후보"] = df["후보"].map(lambda s: " ".join(NAME_OF.get(c, c) for c in s.split()))
pd.set_option("display.max_rows", 200)
df[["이름", "결과", "단계", "확신도", "후보", "검색", "Jev", "메모"]]

## 3. 단계별 · 결과별 요약

In [ ]:
print(f"총 {len(df)}건 · 자동 분류 {(df['결과'] != '확인 필요').sum()}건 · 확인 필요 {(df['결과'] == '확인 필요').sum()}건")
pd.crosstab(df["단계"], df["결과"], margins=True, margins_name="합계")

## 4. 확인 필요 거래와 이유
`메모`: `store_mismatch` = 검색 첫 결과가 다른 가게, `dropped_industry` = 가게가 아닌 업종(금융 등)이라 근거에서 뺌

In [ ]:
df[df["결과"] == "확인 필요"][["이름", "단계", "후보", "검색", "Jev", "메모", "이유"]]

## 5. 정답을 붙여 정확도 보기
`real_merchants_result.csv`를 엑셀로 열어 `정답` 칸에 카테고리 코드(`FOOD`, `ETC` 등)를 채우고 저장한 뒤 실행.
소비가 아닌 거래(이자·ATM 등)는 비워 두면 계산에서 빠진다. 여기 숫자로 SVM 0.85 · Jev 0.9 임계값을 다시 정한다

In [ ]:
raw = pd.read_csv(RESULT, encoding="utf-8-sig", dtype=str).fillna("")
labeled = raw[raw["정답"] != ""]
auto = labeled[labeled["결과"] != "UNCLASSIFIED"]
if labeled.empty:
    print("정답 칸이 비어 있어요. CSV에 정답을 채운 뒤 다시 실행하세요.")
else:
    print(f"정답 {len(labeled)}건 중 자동 분류 {len(auto)}건 ({len(auto) / len(labeled):.0%})")
    print(f"자동 분류 정확도: {(auto['결과'] == auto['정답']).mean():.1%}")
    display(auto.assign(맞음=auto["결과"] == auto["정답"]).groupby("단계")["맞음"].agg(["count", "mean"]))
    display(auto[auto["결과"] != auto["정답"]][["이름", "결과", "정답", "단계", "확신도", "검색", "Jev"]])